# MediFlow AI — XGBoost Classifier Model Training

This notebook contains the local training and evaluation pipeline for the **XGBoost Admission Classifier**.

In [ ]:
import os
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix

print("XGBoost version:", xgb.__version__)

In [ ]:
# Load processed dataset
PROCESSED_CLASSIFICATION_CSV = "../../data/processed/processed_dataset.csv"
if not os.path.exists(PROCESSED_CLASSIFICATION_CSV):
    PROCESSED_CLASSIFICATION_CSV = "data/processed/processed_dataset.csv"

df = pd.read_csv(PROCESSED_CLASSIFICATION_CSV)
df["parsed_timestamp"] = pd.to_datetime(df["parsed_timestamp"])
df = df.sort_values(by="parsed_timestamp").reset_index(drop=True)
print("Loaded dataset shape:", df.shape)

In [ ]:
# Feature columns matching inference schema
feature_cols = [
    "age", "gender", "emergency_severity_level", "hour", "day_of_week",
    "is_weekend", "wait_time", "department", "icu_beds_available",
    "ambulance_requests", "doctor_availability", "oxygen_utilization"
]

# Map department category to integer indexes
depts = ["Self-Referral", "Cardiology", "ICU", "Emergency", "Orthopedics", "Pediatrics"]
df["department"] = df["department"].apply(lambda d: depts.index(d) if d in depts else 0)

X = df[feature_cols]
y = df["admission_target"]

# Temporal Split (75% train, 25% test)
split_idx = int(len(df) * 0.75)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

In [ ]:
# Train Model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    device="cpu"
)
model.fit(X_train, y_train)

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== XGBoost Evaluation Results ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:       {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score:  {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save Model
joblib.dump(model, "model_xgb.pkl")
print("Model successfully saved to 'model_xgb.pkl'.")